# Bilbo — CNN + LSTM Parallel (Bagging) para Detección DGA
**Referencia:** Higham et al., *Real-Time Detection of Dictionary DGA Network Traffic Using Deep Learning*, SN Computer Science 2021.

**Arquitectura:**
- Embedding de caracteres (aprendido durante entrenamiento)
- Rama LSTM: 256 nodos
- Rama CNN: filtros de tamaño {2,3,4,5,6}, 60 filtros cada uno → Global Max Pooling
- ANN (100 neuronas) sobre la concatenación LSTM+CNN → sigmoid
- Clasificación binaria: DGA vs legit

**Entrenamiento:** train_1M.csv | **Evaluación:** 54 familias × 30 runs (archivos pre-generados)

In [1]:
# ── Verificar GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

CUDA disponible: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# ── Montar Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — Ajusta estas rutas antes de correr
# ══════════════════════════════════════════════════════════════════════════════
from pathlib import Path

DATASET_PATH      = Path('/content/drive/MyDrive/NEW_DGA_DETECTOR/train_1M.csv')
OUTPUT_DIR        = Path('/content/drive/MyDrive/NEW_DGA_DETECTOR/models/bilbo-run-001')
FAMILIAS_TEST_DIR = '/content/drive/My Drive/Familias_Test'  # misma carpeta que usa el CNN
RESULTS_DIR       = Path('/content/drive/MyDrive/NEW_DGA_DETECTOR/results/bilbo')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Hiperparámetros (Higham et al. 2021)
MAXLEN       = 75          # longitud máxima del dominio (paper: 63; proyecto usa 75)
EMBED_DIM    = 32
LSTM_SIZE    = 256         # paper: 256
CNN_FILTERS  = [2,3,4,5,6] # paper: filtros de tamaño 2-6
N_FILTERS    = 60          # paper: 60 filtros por tamaño
ANN_HIDDEN   = 100         # paper: 100 neuronas en la capa ANN de agregación
BATCH_SIZE   = 512         # paper: 512
EPOCHS       = 10          # paper: 10
LR           = 1e-3        # Adam
TEST_SIZE    = 0.2
RANDOM_STATE = 42
RUNS         = 30          # runs por familia en evaluación

print('Config OK')
print(f'Dataset:       {DATASET_PATH}')
print(f'Output:        {OUTPUT_DIR}')
print(f'Test familias: {FAMILIAS_TEST_DIR}')

Config OK
Dataset:       /content/drive/MyDrive/NEW_DGA_DETECTOR/train_1M.csv
Output:        /content/drive/MyDrive/NEW_DGA_DETECTOR/models/bilbo-run-001
Test familias: /content/drive/My Drive/Familias_Test


In [4]:
# ── Imports ────────────────────────────────────────────────────────────────────
import re
import json
import time
import string
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, confusion_matrix, classification_report)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


In [5]:
# ── Tokenizador de caracteres (consistente con proyecto) ───────────────────────
CHARS   = string.ascii_lowercase + string.digits + '-._'
CHAR2IDX = {c: i + 1 for i, c in enumerate(CHARS)}  # 0 = padding
VOCAB_SIZE = len(CHARS) + 1  # 40

def encode_domain(domain: str) -> list:
    domain = str(domain).lower().strip()
    encoded = [CHAR2IDX.get(c, 0) for c in domain[:MAXLEN]]
    # Left-padding (como en Higham 2021)
    pad_len = MAXLEN - len(encoded)
    return [0] * pad_len + encoded

# Verificación
test_enc = encode_domain('google.com')
print(f'Vocab size: {VOCAB_SIZE}')
print(f'Encoded "google.com" (últimos 10): {test_enc[-10:]}')

Vocab size: 40
Encoded "google.com" (últimos 10): [7, 15, 15, 7, 12, 5, 38, 3, 15, 13]


In [6]:
# ── Carga y limpieza de datos ──────────────────────────────────────────────────
DOMAIN_RE = re.compile(r'^[a-z0-9.-]+$')

def normalize_domain(value: str) -> str:
    from urllib.parse import urlparse
    domain = str(value).strip().lower()
    parsed = urlparse(domain)
    if parsed.netloc:
        domain = parsed.netloc
    domain = domain.split('@')[-1].split('/')[0].split(':')[0].rstrip('.')
    domain = re.sub(r'\s+', '', domain)
    if not DOMAIN_RE.match(domain):
        domain = re.sub(r'[^a-z0-9.-]', '', domain)
    return domain

t0 = time.time()
df_raw = pd.read_csv(DATASET_PATH)
print(f'Filas brutas: {len(df_raw):,}  ({time.time()-t0:.1f}s)')

# Limpiar
df = pd.DataFrame()
df['domain'] = df_raw['domain'].map(normalize_domain)
df['label']  = (df_raw['label'].str.strip().str.lower() == 'dga').astype(int)
df['family'] = df_raw['family'].astype(str).str.strip().str.lower()
df = df[df['domain'].ne('')].drop_duplicates(subset=['domain', 'label']).reset_index(drop=True)

print(f'Filas limpias: {len(df):,}')
print(f'Labels: {df["label"].value_counts().to_dict()}')
print(f'Familias: {df["family"].nunique()}')
df.head()

Filas brutas: 1,080,000  (1.4s)
Filas limpias: 1,053,080
Labels: {0: 540000, 1: 513080}
Familias: 55


,domain,label,family
0,newmedialove.ru,0,legit
1,bankesb.com,0,legit
2,sbjnvufhillsszger.net,1,fobber
3,rpltm.online,0,legit
4,theblotsays.com,0,legit


In [7]:
# ── Split honesto por familia (igual que notebooks Logit/ModernBERT) ───────────
dga_df    = df[df['label'] == 1].reset_index(drop=True)
benign_df = df[df['label'] == 0].reset_index(drop=True)

# DGA: GroupShuffleSplit por familia (familias completas en train o test)
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
dga_train_idx, dga_test_idx = next(
    splitter.split(dga_df['domain'], dga_df['label'], groups=dga_df['family'])
)
dga_train = dga_df.iloc[dga_train_idx]
dga_test  = dga_df.iloc[dga_test_idx]

# Legit: split aleatorio
b_train_idx, b_test_idx = train_test_split(
    benign_df.index, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
benign_train = benign_df.loc[b_train_idx]
benign_test  = benign_df.loc[b_test_idx]

train_df = pd.concat([dga_train, benign_train]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_df  = pd.concat([dga_test,  benign_test ]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')
print(f'Train labels: {train_df["label"].value_counts().to_dict()}')
print(f'Train familias DGA: {train_df[train_df["label"]==1]["family"].nunique()}')
print(f'Test  familias DGA: {test_df[test_df["label"]==1]["family"].nunique()}')

Train: 845,639 | Test: 207,441
Train labels: {0: 432000, 1: 413639}
Train familias DGA: 43
Test  familias DGA: 11


In [8]:
# ── PyTorch Dataset & DataLoader ───────────────────────────────────────────────
class DGADataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.X = torch.tensor(
            [encode_domain(d) for d in df['domain']], dtype=torch.long
        )
        self.y = torch.tensor(df['label'].values, dtype=torch.float32)

    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_ds = DGADataset(train_df)
test_ds  = DGADataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Batches train: {len(train_loader)} | Batches test: {len(test_loader)}')

Batches train: 1652 | Batches test: 406


In [9]:
# ── Modelo Bilbo: CNN + LSTM en paralelo (Higham et al. 2021) ─────────────────
class BilboModel(nn.Module):
    """
    Arquitectura 'bagging':
    - Rama LSTM(256) sobre embeddings de caracteres
    - Rama CNN con filtros {2,3,4,5,6} × 60 + Global Max Pooling
    - Concatenación → ANN(100) → sigmoid
    """
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)

        # Rama LSTM
        self.lstm = nn.LSTM(EMBED_DIM, LSTM_SIZE, batch_first=True)

        # Rama CNN: un Conv1d por cada tamaño de filtro
        self.convs = nn.ModuleList([
            nn.Conv1d(EMBED_DIM, N_FILTERS, kernel_size=k, padding=k // 2)
            for k in CNN_FILTERS
        ])

        # ANN de agregación (bagging)
        cnn_out_dim = N_FILTERS * len(CNN_FILTERS)   # 60 × 5 = 300
        combined_dim = LSTM_SIZE + cnn_out_dim        # 256 + 300 = 556
        self.ann = nn.Sequential(
            nn.Linear(combined_dim, ANN_HIDDEN),
            nn.ReLU(),
            nn.Linear(ANN_HIDDEN, 1)
        )

    def forward(self, x):
        emb = self.embedding(x)          # (B, L, E)

        # Rama LSTM
        _, (h, _) = self.lstm(emb)       # h: (1, B, LSTM_SIZE)
        lstm_feat = h.squeeze(0)         # (B, LSTM_SIZE)

        # Rama CNN
        emb_t = emb.transpose(1, 2)      # (B, E, L)
        cnn_feats = []
        for conv in self.convs:
            c = torch.relu(conv(emb_t))  # (B, N_FILTERS, L')
            c = c.max(dim=2)[0]          # Global max pooling → (B, N_FILTERS)
            cnn_feats.append(c)
        cnn_feat = torch.cat(cnn_feats, dim=1)   # (B, 300)

        # Bagging: concatenar y clasificar
        combined = torch.cat([lstm_feat, cnn_feat], dim=1)  # (B, 556)
        return self.ann(combined).squeeze(1)                 # (B,)


model = BilboModel().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {total_params:,}')
print(model)

Parámetros entrenables: 392,741
BilboModel(
  (embedding): Embedding(40, 32, padding_idx=0)
  (lstm): LSTM(32, 256, batch_first=True)
  (convs): ModuleList(
    (0): Conv1d(32, 60, kernel_size=(2,), stride=(1,), padding=(1,))
    (1): Conv1d(32, 60, kernel_size=(3,), stride=(1,), padding=(1,))
    (2): Conv1d(32, 60, kernel_size=(4,), stride=(1,), padding=(2,))
    (3): Conv1d(32, 60, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): Conv1d(32, 60, kernel_size=(6,), stride=(1,), padding=(3,))
  )
  (ann): Sequential(
    (0): Linear(in_features=556, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=1, bias=True)
  )
)


In [10]:
# ── Entrenamiento ──────────────────────────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scaler    = GradScaler()  # Mixed precision para T4

best_val_f1  = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    t_epoch = time.time()

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with autocast():
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── Validación ─────────────────────────────────────────────────────────────
    model.eval()
    all_preds, all_labels = [], []
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(X_batch)
                loss   = criterion(logits, y_batch)
            val_loss += loss.item()
            preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.long().cpu().numpy())

    val_loss /= len(test_loader)
    val_f1 = f1_score(all_labels, all_preds, zero_division=0)
    val_acc = accuracy_score(all_labels, all_preds)
    elapsed = time.time() - t_epoch

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
          f'val_f1={val_f1:.4f} | val_acc={val_acc:.4f} | {elapsed:.1f}s')

    history.append({'epoch': epoch, 'train_loss': train_loss,
                    'val_loss': val_loss, 'val_f1': val_f1, 'val_acc': val_acc})

    # Guardar mejor modelo
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), OUTPUT_DIR / 'bilbo_best.pth')
        print(f'  ✓ Mejor modelo guardado (F1={best_val_f1:.4f})')

print(f'\nEntrenamiento completado. Mejor val_F1={best_val_f1:.4f}')

/tmp/ipykernel_551/1021878464.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()  # Mixed precision para T4
/tmp/ipykernel_551/1021878464.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_551/1021878464.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 01/10 | train_loss=0.2003 | val_loss=0.2945 | val_f1=0.8950 | val_acc=0.9042 | 38.4s
  ✓ Mejor modelo guardado (F1=0.8950)


/tmp/ipykernel_551/1021878464.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_551/1021878464.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 02/10 | train_loss=0.1383 | val_loss=0.3367 | val_f1=0.8832 | val_acc=0.8975 | 34.0s


/tmp/ipykernel_551/1021878464.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_551/1021878464.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

Epoch 08/10 | train_loss=0.0717 | val_loss=0.3537 | val_f1=0.8760 | val_acc=0.8898 | 33.5s


/tmp/ipykernel_551/1021878464.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_551/1021878464.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 09/10 | train_loss=0.0640 | val_loss=0.5024 | val_f1=0.8605 | val_acc=0.8788 | 33.0s


/tmp/ipykernel_551/1021878464.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_551/1021878464.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 10/10 | train_loss=0.0576 | val_loss=0.4943 | val_f1=0.8563 | val_acc=0.8737 | 34.8s

Entrenamiento completado. Mejor val_F1=0.8950


In [11]:
# ── Guardar modelo final + métricas de entrenamiento ──────────────────────────
# Cargar el mejor checkpoint
model.load_state_dict(torch.load(OUTPUT_DIR / 'bilbo_best.pth', map_location=DEVICE))
model.eval()

# Reporte final en test set
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(DEVICE))
        preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.long().numpy())

report = classification_report(all_labels, all_preds, digits=4)
metrics = {
    'model': 'Bilbo',
    'reference': 'Higham et al. 2021',
    'train_rows': len(train_df),
    'test_rows': len(test_df),
    'best_val_f1': best_val_f1,
    'final_f1': float(f1_score(all_labels, all_preds, zero_division=0)),
    'final_precision': float(precision_score(all_labels, all_preds, zero_division=0)),
    'final_recall': float(recall_score(all_labels, all_preds, zero_division=0)),
    'final_accuracy': float(accuracy_score(all_labels, all_preds)),
    'history': history
}

(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
(OUTPUT_DIR / 'classification_report.txt').write_text(report)

print(report)
print(f'Guardado en: {OUTPUT_DIR}')

              precision    recall  f1-score   support

           0     0.8749    0.9521    0.9119    108000
           1     0.9425    0.8521    0.8950     99441

    accuracy                         0.9042    207441
   macro avg     0.9087    0.9021    0.9034    207441
weighted avg     0.9073    0.9042    0.9038    207441

Guardado en: /content/drive/MyDrive/NEW_DGA_DETECTOR/models/bilbo-run-001


In [12]:
# ── Helpers para evaluación por familia ───────────────────────────────────────
def predict_domains(domains_series: pd.Series):
    """Recibe una Serie de dominios, devuelve predicciones binarias (numpy)."""
    model.eval()
    encoded = torch.tensor(
        [encode_domain(d) for d in domains_series], dtype=torch.long
    )
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(encoded), 256):
            batch = encoded[i:i+256].to(DEVICE)
            logits = model(batch)
            preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

def fpr_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return fpr, tpr

print('Helpers cargados')

Helpers cargados


In [13]:
# ── Evaluación por familia (54 familias × 30 runs) ────────────────────────────
# Mismo protocolo que CNN: 50 DGA + 50 legit por run, leyendo de Familias_Test
FAMILIES = [
    'alureon.gz','bamital.gz','banjori.gz','bedep.gz','charbot.gz','chinad.gz',
    'conficker.gz','corebot.gz','cryptolocker.gz','deception.gz','dircrypt.gz',
    'dnschanger.gz','dyre.gz','emotet.gz','fobber.gz','gameover.gz','gozi.gz',
    'kraken.gz','locky.gz','manuelita.gz','matsnu.gz','monerominer.gz','murofet.gz',
    'murofetweekly.gz','mydoom.gz','necurs.gz','nymaim.gz','oderoor.gz','padcrypt.gz',
    'pitou.gz','proslikefan.gz','pushdo.gz','pykspa.gz','qadars.gz','qakbot.gz',
    'qsnatch.gz','ramdo.gz','ramnit.gz','ranbyus.gz','rovnix.gz','shiotob.gz',
    'simda.gz','sisron.gz','sphinx.gz','suppobox.gz','symmi.gz','tempedreve.gz',
    'tinba.gz','tinynuke.gz','vawtrak.gz','vidro.gz','virut.gz','zeus-newgoz.gz',
    'zloader.gz'
]

results = []

for family in FAMILIES:
    acc_l, pre_l, rec_l, f1_l = [], [], [], []
    fpr_l, tpr_l, qt_l = [], [], []

    # Un reader por familia — chunksize=50 → run 0 lee filas 0-49, run 1 lee 50-99, etc.
    # Igual que CNN: los mismos dominios en cada run para comparación justa
    try:
        dga_reader   = pd.read_csv(f'{FAMILIAS_TEST_DIR}/{family}',   chunksize=50)
        legit_reader = pd.read_csv(f'{FAMILIAS_TEST_DIR}/legit.gz',   chunksize=50)
    except FileNotFoundError as e:
        print(f'[WARN] Archivo no encontrado: {e}')
        continue

    for run in range(RUNS):
        try:
            dga_chunk   = dga_reader.get_chunk()
            legit_chunk = legit_reader.get_chunk()
        except StopIteration:
            print(f'[WARN] {family}: sin suficientes chunks en run {run}')
            break

        df_run = pd.concat([dga_chunk, legit_chunk], ignore_index=True)
        y_true = (df_run['label'] == 'dga').astype(int).values

        t0 = time.perf_counter()
        y_pred = predict_domains(df_run['domain'])
        elapsed = time.perf_counter() - t0
        per_domain_s = elapsed / len(df_run)

        acc_l.append(accuracy_score(y_true, y_pred))
        pre_l.append(precision_score(y_true, y_pred, zero_division=0))
        rec_l.append(recall_score(y_true, y_pred, zero_division=0))
        f1_l.append(f1_score(y_true, y_pred, zero_division=0))
        fpr_v, tpr_v = fpr_tpr(y_true, y_pred)
        fpr_l.append(fpr_v)
        tpr_l.append(tpr_v)
        qt_l.append(per_domain_s)

    name = family.split('.')[0]
    print(f'{name:20}: acc={np.mean(acc_l):.3f} f1={np.mean(f1_l):.3f} '
          f'pre={np.mean(pre_l):.3f} rec={np.mean(rec_l):.3f} '
          f'FPR={np.mean(fpr_l):.3f} TPR={np.mean(tpr_l):.3f}')

    results.append({
        'Family':            name,
        'Accuracy Mean':     np.mean(acc_l),   'Accuracy Std Dev':   np.std(acc_l),
        'F1 Score Mean':     np.mean(f1_l),    'F1 Score Std Dev':   np.std(f1_l),
        'Precision Mean':    np.mean(pre_l),   'Precision Std Dev':  np.std(pre_l),
        'Recall Mean':       np.mean(rec_l),   'Recall Std Dev':     np.std(rec_l),
        'FPR Mean':          np.mean(fpr_l),   'FPR Std Dev':        np.std(fpr_l),
        'TPR Mean':          np.mean(tpr_l),   'TPR Std Dev':        np.std(tpr_l),
        'Query Time Mean':   np.mean(qt_l),    'Query Time Std Dev': np.std(qt_l),
    })

print('\nEvaluación completada.')

alureon             : acc=0.954 f1=0.954 pre=0.948 rec=0.961 FPR=0.054 TPR=0.961
bamital             : acc=0.973 f1=0.974 pre=0.950 rec=1.000 FPR=0.054 TPR=1.000
banjori             : acc=0.903 f1=0.898 pre=0.942 rec=0.860 FPR=0.054 TPR=0.860
bedep               : acc=0.966 f1=0.966 pre=0.949 rec=0.985 FPR=0.054 TPR=0.985
charbot             : acc=0.735 f1=0.660 pre=0.906 rec=0.523 FPR=0.054 TPR=0.523
chinad              : acc=0.973 f1=0.974 pre=0.950 rec=0.999 FPR=0.054 TPR=0.999
conficker           : acc=0.907 f1=0.903 pre=0.942 rec=0.868 FPR=0.054 TPR=0.868
corebot             : acc=0.973 f1=0.974 pre=0.950 rec=1.000 FPR=0.054 TPR=1.000
cryptolocker        : acc=0.961 f1=0.961 pre=0.948 rec=0.975 FPR=0.054 TPR=0.975
deception           : acc=0.970 f1=0.971 pre=0.949 rec=0.993 FPR=0.054 TPR=0.993
dircrypt            : acc=0.961 f1=0.962 pre=0.948 rec=0.976 FPR=0.054 TPR=0.976
dnschanger          : acc=0.953 f1=0.953 pre=0.947 rec=0.960 FPR=0.054 TPR=0.960
dyre                : acc=0.

In [14]:
# ── Guardar resultados y promedios finales ─────────────────────────────────────
df_results = pd.DataFrame(results)

results_csv = RESULTS_DIR / 'df_results_Bilbo.csv'
df_results.to_csv(results_csv, index=False)
print(f'Resultados guardados: {results_csv}')

# Métricas promedio globales
avg_cols = ['Accuracy Mean','F1 Score Mean','Precision Mean','Recall Mean',
            'FPR Mean','TPR Mean','Query Time Mean']
avg = df_results[avg_cols].mean().to_dict()

final_metrics = {
    'model': 'Bilbo',
    'reference': 'Higham et al., SN Computer Science 2021',
    **avg
}
final_json = RESULTS_DIR / 'final_test_metrics_Bilbo.json'
final_json.write_text(json.dumps(final_metrics, indent=2))

print('\n── Métricas promedio (54 familias) ──')
for k, v in avg.items():
    print(f'  {k}: {v:.4f}')

display(df_results)

Resultados guardados: /content/drive/MyDrive/NEW_DGA_DETECTOR/results/bilbo/df_results_Bilbo.csv

── Métricas promedio (54 familias) ──
  Accuracy Mean: 0.9207
  F1 Score Mean: 0.9000
  Precision Mean: 0.9303
  Recall Mean: 0.8954
  FPR Mean: 0.0540
  TPR Mean: 0.8954
  Query Time Mean: 0.0001


,Family,Accuracy Mean,Accuracy Std Dev,F1 Score Mean,F1 Score Std Dev,Precision Mean,Precision Std Dev,Recall Mean,Recall Std Dev,FPR Mean,FPR Std Dev,TPR Mean,TPR Std Dev,Query Time Mean,Query Time Std Dev
0,alureon,0.953667,0.020410,0.954004,0.020221,0.947631,0.027758,0.961333,0.029181,0.054,0.029732,0.961333,0.029181,0.000077,3.513371e-06
1,bamital,0.973000,0.014866,0.973913,0.014034,0.949516,0.026570,1.000000,0.000000,0.054,0.029732,1.000000,0.000000,0.000075,2.141424e-06
2,banjori,0.903000,0.026602,0.898343,0.028710,0.941677,0.030875,0.860000,0.041633,0.054,0.029732,0.860000,0.041633,0.000074,2.559570e-06
3,bedep,0.965667,0.015424,0.966451,0.014819,0.948903,0.026715,0.985333,0.017839,0.054,0.029732,0.985333,0.017839,0.000076,2.560564e-06
4,charbot,0.734667,0.048009,0.659758,0.076825,0.906120,0.050711,0.523333,0.087496,0.054,0.029732,0.523333,0.087496,0.000076,2.229466e-06
5,chinad,0.972667,0.014591,0.973576,0.013760,0.949503,0.026555,0.999333,0.003590,0.054,0.029732,0.999333,0.003590,0.000073,2.456298e-06
6,conficker,0.907000,0.030676,0.902688,0.033793,0.942031,0.030648,0.868000,0.050226,0.054,0.029732,0.868000,0.050226,0.000079,1.967769e-06
7,corebot,0.973000,0.014866,0.973913,0.014034,0.949516,0.026570,1.000000,0.000000,0.054,0.029732,1.000000,0.000000,0.000083,3.091671e-06
8,cryptolocker,0.960667,0.020806,0.961333,0.020275,0.948143,0.027798,0.975333,0.021092,0.054,0.029732,0.975333,0.021092,0.000089,1.027681e-05
9,deception,0.969667,0.014941,0.970535,0.014199,0.949247,0.026601,0.993333,0.011926,0.054,0.029732,0.993333,0.011926,0.000080,4.462401e-06


In [ ]:
# ── Descargar archivos localmente ─────────────────────────────────────────────
import shutil
from google.colab import files

for src in [results_csv, final_json, OUTPUT_DIR / 'metrics.json']:
    dst = Path('/content') / src.name
    shutil.copy(src, dst)
    files.download(str(dst))
    print(f'Descargando: {src.name}')